# Cross-Fitting: Eliminating Overfitting Bias

**Goal:** See the overfitting bias from in-sample DML and how K-fold cross-fitting eliminates it.

**Time:** 15 minutes

You will compare in-sample predictions vs cross-fitted predictions, observe the bias,
and compare DML1 vs DML2 aggregation methods.

In [ ]:
import sys; sys.path.insert(0, '../../../../..')
from resources.notebook_style import apply_course_theme, learning_objectives, section_divider, callout, key_takeaways
from resources.graphics.plot_theme import apply_plot_theme

In [ ]:
# Apply course styling
apply_course_theme()
apply_plot_theme()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import KFold

np.random.seed(42)
plt.style.use('seaborn-v0_8-darkgrid')

## Setup

Simple DGP with known ground truth.

In [ ]:
n = 2000
p = 50
true_theta = 1.0

X = np.random.randn(n, p)
D = 0.5 * X[:, 0] + 0.3 * X[:, 1] + np.random.randn(n) * 0.5
Y = true_theta * D + X[:, 0] + 0.5 * X[:, 2] + np.random.randn(n)

print(f'n={n}, p={p}, true_theta={true_theta}')

## In-Sample vs Cross-Fitted: Single Run

Compare the two approaches on the same dataset.

In [ ]:
# In-sample (WRONG)
rf_y = RandomForestRegressor(n_estimators=200, random_state=42).fit(X, Y)
rf_d = RandomForestRegressor(n_estimators=200, random_state=42).fit(X, D)

rY_in = Y - rf_y.predict(X)
rD_in = D - rf_d.predict(X)
theta_in = np.sum(rD_in * rY_in) / np.sum(rD_in ** 2)

# Cross-fitted (CORRECT)
rY_cf = np.zeros(n)
rD_cf = np.zeros(n)
kf = KFold(n_splits=5, shuffle=True, random_state=42)

for train_idx, test_idx in kf.split(X):
    rf_y_k = RandomForestRegressor(n_estimators=200, random_state=42)
    rf_d_k = RandomForestRegressor(n_estimators=200, random_state=42)
    rf_y_k.fit(X[train_idx], Y[train_idx])
    rf_d_k.fit(X[train_idx], D[train_idx])
    rY_cf[test_idx] = Y[test_idx] - rf_y_k.predict(X[test_idx])
    rD_cf[test_idx] = D[test_idx] - rf_d_k.predict(X[test_idx])

theta_cf = np.sum(rD_cf * rY_cf) / np.sum(rD_cf ** 2)

print(f'True effect:           {true_theta:.2f}')
print(f'In-sample DML:         {theta_in:.2f}  (biased!)')
print(f'Cross-fitted DML:      {theta_cf:.2f}  (correct!)')
print(f'\nMean |resid_D| in-sample:    {np.mean(np.abs(rD_in)):.4f}')
print(f'Mean |resid_D| cross-fitted: {np.mean(np.abs(rD_cf)):.4f}')
print(f'\nIn-sample residuals are {np.mean(np.abs(rD_cf))/np.mean(np.abs(rD_in)):.1f}x smaller — this causes the bias.')

## Monte Carlo: Distribution of Estimates

Run 100 simulations to see the full distribution of both approaches.

In [ ]:
n_sims = 100
insample_ests = []
crossfit_ests = []

for sim in range(n_sims):
    rng = np.random.RandomState(sim)
    X_s = rng.randn(n, p)
    D_s = 0.5 * X_s[:, 0] + 0.3 * X_s[:, 1] + rng.randn(n) * 0.5
    Y_s = true_theta * D_s + X_s[:, 0] + 0.5 * X_s[:, 2] + rng.randn(n)

    # In-sample
    rf_y = RandomForestRegressor(100, random_state=sim).fit(X_s, Y_s)
    rf_d = RandomForestRegressor(100, random_state=sim).fit(X_s, D_s)
    rY = Y_s - rf_y.predict(X_s)
    rD = D_s - rf_d.predict(X_s)
    insample_ests.append(np.sum(rD * rY) / np.sum(rD ** 2))

    # Cross-fitted
    rY_c, rD_c = np.zeros(n), np.zeros(n)
    for tr, te in KFold(5, shuffle=True, random_state=sim).split(X_s):
        rf_y_k = RandomForestRegressor(100, random_state=sim).fit(X_s[tr], Y_s[tr])
        rf_d_k = RandomForestRegressor(100, random_state=sim).fit(X_s[tr], D_s[tr])
        rY_c[te] = Y_s[te] - rf_y_k.predict(X_s[te])
        rD_c[te] = D_s[te] - rf_d_k.predict(X_s[te])
    crossfit_ests.append(np.sum(rD_c * rY_c) / np.sum(rD_c ** 2))

print(f'In-sample:   mean={np.mean(insample_ests):.3f}, std={np.std(insample_ests):.3f}')
print(f'Cross-fitted: mean={np.mean(crossfit_ests):.3f}, std={np.std(crossfit_ests):.3f}')
print(f'True: {true_theta:.2f}')

## Visualise: Distribution Comparison

Histogram showing the sampling distributions of both estimators.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

ax.hist(insample_ests, bins=25, alpha=0.5, color='crimson', label='In-sample (biased)', density=True)
ax.hist(crossfit_ests, bins=25, alpha=0.5, color='forestgreen', label='Cross-fitted (correct)', density=True)
ax.axvline(x=true_theta, color='black', linestyle='--', linewidth=2, label=f'True = {true_theta}')
ax.set_xlabel('Treatment Effect Estimate', fontsize=12)
ax.set_ylabel('Density', fontsize=12)
ax.set_title('Cross-Fitting Eliminates Overfitting Bias', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f'In-sample bias: {np.mean(insample_ests) - true_theta:+.3f}')
print(f'Cross-fitted bias: {np.mean(crossfit_ests) - true_theta:+.3f}')

## DML1 vs DML2 Comparison

Compare the two aggregation methods across simulations.

In [ ]:
dml1_ests = []
dml2_ests = []

for sim in range(n_sims):
    rng = np.random.RandomState(sim)
    X_s = rng.randn(n, p)
    D_s = 0.5 * X_s[:, 0] + 0.3 * X_s[:, 1] + rng.randn(n) * 0.5
    Y_s = true_theta * D_s + X_s[:, 0] + 0.5 * X_s[:, 2] + rng.randn(n)

    rY_c, rD_c = np.zeros(n), np.zeros(n)
    fold_thetas = []
    kf_sim = KFold(5, shuffle=True, random_state=sim)

    for tr, te in kf_sim.split(X_s):
        rf_y_k = RandomForestRegressor(100, random_state=sim).fit(X_s[tr], Y_s[tr])
        rf_d_k = RandomForestRegressor(100, random_state=sim).fit(X_s[tr], D_s[tr])
        rY_c[te] = Y_s[te] - rf_y_k.predict(X_s[te])
        rD_c[te] = D_s[te] - rf_d_k.predict(X_s[te])
        fold_thetas.append(np.sum(rD_c[te] * rY_c[te]) / np.sum(rD_c[te] ** 2))

    dml1_ests.append(np.mean(fold_thetas))
    dml2_ests.append(np.sum(rD_c * rY_c) / np.sum(rD_c ** 2))

print(f'DML1: mean={np.mean(dml1_ests):.4f}, std={np.std(dml1_ests):.4f}')
print(f'DML2: mean={np.mean(dml2_ests):.4f}, std={np.std(dml2_ests):.4f}')
print(f'\nDML2 has {np.std(dml1_ests)/np.std(dml2_ests):.2f}x lower variance than DML1.')

## Summary

**What you learned:**
1. In-sample ML predictions create overfitting bias that inflates treatment effect estimates
2. K-fold cross-fitting eliminates this bias by ensuring all predictions are out-of-sample
3. DML2 (pool all residuals) is preferred over DML1 (average fold estimates) due to lower variance
4. K=5 folds is the standard choice, with diminishing returns beyond K=10

**What is next:**
- **Module 05:** Putting it all together with the `doubleml` library
- **Module 06:** Interactive regression models for binary treatments

**Key takeaway:** Cross-fitting is mandatory, not optional. Without it, DML is biased.

In [ ]:
learning_objectives(["In-sample ML predictions create overfitting bias that inflates treatment effect estimates", "K-fold cross-fitting eliminates this bias by ensuring all predictions are out-of-sample", "DML2 (pool all residuals) is preferred over DML1 (average fold estimates) due to lower variance", "K=5 folds is the standard choice, with diminishing returns beyond K=10"])

In [ ]:
key_takeaways([
    "Core concept from this notebook demonstrated with working code",
    "Key parameters and their effects on results",
    "When to apply this technique vs alternatives"
])